# Mancino Market: Predicting Daily Unit Sales
## Complete regression tree exercise | Business Analytics and Data Visualization
Prof. Jose Mendoza · Fall 2026

How many units will a store sell in a product category on a future day? Use a regression tree to predict daily unit sales, compare different tree depths, and evaluate predictions against a mean-sales baseline. The outcome is a numeric quantity, not a class label.

Mancino Market is a synthetic grocery retailer with five fictional Manhattan stores. One row represents one date, store, and product category. Predictions can inform replenishment planning, but observed sales do not measure unmet demand during stockouts.

### How to use this notebook
1. Upload this `.ipynb` file at [Google Colab](https://colab.research.google.com/) using **File > Upload notebook**. A standard CPU runtime is sufficient.
2. Run sections 1–5 to prepare the data and baseline.
3. Run section 6 to fit regression trees with maximum depths of 3, 6, and 12.
4. Compare validation results in section 7, then interpret the selected tree in section 8.
5. When selection is complete, change `RUN_FINAL_TEST` to `True` in section 9 and run the final test once.

All code is complete. No separate CSV upload is needed when the GitHub connection is available. The data, predictors, and date splits match the companion KNN exercise.


## 1. Import the tools
Colab includes these libraries. The notebook prints their versions so results can be traced to the environment used.


In [ ]:
import io
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False,
                     'axes.spines.right': False})
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


## 2. Load and inspect the data
Source: [mancino_daily_sales.csv](https://github.com/jrmst102/mancino/blob/main/data/v1_2025-11-01/mancino_daily_sales.csv).
The verified file contains 22,875 rows, covering January 1–November 1, 2025. The code reports the current file dimensions in case the repository changes.

| Field | Meaning | Use in this notebook |
|---|---|---|
| `date` | Selling date | Split data and derive calendar features |
| `store_id` | Store identifier | Categorical predictor |
| `category` | Product category | Categorical predictor |
| `day_of_week` | Weekday | Recomputed from the date |
| `avg_price` | Observed daily average price | Excluded |
| `units_sold` | Daily units for a store and category | Target |
| `revenue` | Realized sales revenue | Excluded |

Revenue contains information about the sales outcome and would leak information into the model. The observed average price is also excluded because its availability before the selling day has not been established. A verified, planned price could be added in a later exercise.


In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/jrmst102/mancino/main/data/v1_2025-11-01/mancino_daily_sales.csv'
LOCAL_CSV = Path('mancino_daily_sales.csv')

if LOCAL_CSV.exists():
    df = pd.read_csv(LOCAL_CSV)
    print('Loaded local CSV:', LOCAL_CSV)
else:
    try:
        df = pd.read_csv(DATA_URL)
        print('Loaded CSV from GitHub.')
    except Exception as error:
        print('GitHub download failed. Upload mancino_daily_sales.csv when prompted.')
        from google.colab import files
        uploaded = files.upload()
        if 'mancino_daily_sales.csv' not in uploaded:
            raise ValueError('Please upload the named Mancino CSV.') from error
        df = pd.read_csv(io.BytesIO(uploaded['mancino_daily_sales.csv']))

required = {'date', 'store_id', 'category', 'units_sold'}
assert required.issubset(df.columns), 'The CSV is missing required columns.'
df['date'] = pd.to_datetime(df['date'], errors='raise')
assert not df[list(required)].isna().any().any(), 'Check missing required values.'
assert not df.duplicated(['date', 'store_id', 'category']).any(), 'Check duplicate rows.'
assert pd.api.types.is_numeric_dtype(df['units_sold'])
assert df['units_sold'].ge(0).all(), 'Check negative unit sales.'
df = df.sort_values(['date', 'store_id', 'category']).reset_index(drop=True)
print(f'{len(df):,} rows | {df.store_id.nunique()} stores | {df.category.nunique()} categories')
print('Dates:', df.date.min().date(), 'to', df.date.max().date())
display(df.head())
display(df.isna().sum().rename('Missing values').to_frame())


## 3. Define predictors and split by date
Use only information available before the selling day: store, category, weekday, and day of year. Store numbers are identifiers, so they are treated as categories. Day of year supplies a simple numeric time feature; it is not a full seasonality model.

The earliest 60% of dates form the training set, the next 20% form the validation set, and the latest 20% form the test set. Every observation from a given date stays in the same set. This asks whether patterns learned from earlier dates carry forward to later dates for the existing stores and categories.

Train on the training set, choose settings using validation results, and open the test set only after those choices are complete. This notebook does not use lagged sales, promotions, or inventory data.


In [ ]:
df['store_id'] = df['store_id'].astype(str)
df['weekday'] = df['date'].dt.day_name()
df['day_of_year'] = df['date'].dt.dayofyear
CATEGORICAL = ['store_id', 'category', 'weekday']
NUMERIC = ['day_of_year']
FEATURES = CATEGORICAL + NUMERIC
TARGET = 'units_sold'

dates = np.sort(df['date'].unique())
train_end = dates[int(len(dates) * 0.60) - 1]
validation_end = dates[int(len(dates) * 0.80) - 1]
train = df.loc[df.date <= train_end].copy()
validation = df.loc[(df.date > train_end) & (df.date <= validation_end)].copy()
test = df.loc[df.date > validation_end].copy()
assert train.date.max() < validation.date.min() <= validation.date.max() < test.date.min()

X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = validation[FEATURES], validation[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
display(pd.DataFrame([
    {'Set': name, 'Rows': len(part), 'First date': part.date.min().date(),
     'Last date': part.date.max().date()}
    for name, part in [('Training', train), ('Validation', validation), ('Test', test)]
]))


In [ ]:
# Explore training outcomes only; keep later outcomes for evaluation.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(y_train, bins=30, color='#356DA8', edgecolor='white')
axes[0].set(xlabel='Daily units per store/category', ylabel='Number of observations',
            title='Training sales distribution')
train.groupby('date').units_sold.sum().plot(ax=axes[1], color='#356DA8')
axes[1].set(xlabel='Date', ylabel='Total units across stores/categories',
            title='Training sales over time')
plt.tight_layout()
plt.show()


## 4. Prepare the regression tree workflow
One-hot encoding turns categories into indicator columns. A regression tree makes threshold-based splits, so numeric predictors do not need scaling. Day of year stays in its original units to make the tree diagram easier to read.

Each tree configuration gets a fresh pipeline. The encoder is fitted on training data only, then applied to validation data without refitting. The default squared-error criterion favors splits that reduce variation in unit sales within the resulting groups. Each terminal leaf predicts the mean training sales of observations in that leaf.

| Metric | Interpretation | Preferred direction |
|---|---|---|
| MAE | Average absolute error, in units per store/category/day | Lower |
| RMSE | Error in units, with larger misses receiving more weight | Lower |
| R² | Performance relative to predicting the evaluation set's mean | Higher |

R² can be negative. It is not a percentage of predictions that are correct. Use validation MAE to select tree depth for this exercise. The splitting criterion and the evaluation metric serve different purposes.


In [ ]:
def make_pipeline(estimator):
    preparation = ColumnTransformer([
        ('categories', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
        ('numeric', 'passthrough', NUMERIC)
    ])
    return Pipeline([('prepare', preparation), ('model', estimator)])

def metrics(actual, predicted):
    return {'MAE': mean_absolute_error(actual, predicted),
            'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)}

# Rerunning this cell clears previous model runs.
models, results = {}, {}

def fit_and_evaluate(name, estimator):
    pipeline = make_pipeline(estimator)
    started = time.perf_counter()
    pipeline.fit(X_train, y_train)
    seconds = time.perf_counter() - started
    train_scores = metrics(y_train, pipeline.predict(X_train))
    val_scores = metrics(y_val, pipeline.predict(X_val))
    models[name] = pipeline
    results[name] = {'Model': name, 'Train MAE': train_scores['MAE'],
                     'Validation MAE': val_scores['MAE'],
                     'Validation RMSE': val_scores['RMSE'],
                     'Validation R2': val_scores['R2'], 'Fit seconds': seconds}
    display(pd.DataFrame([results[name]]).round(3))
    return pipeline


## 5. Establish a baseline
The baseline always predicts the training set's mean sales. A more complicated model must earn its place by improving on this simple rule. A later extension could use historical averages for each store, category, and weekday as a stronger baseline.


In [ ]:
baseline = fit_and_evaluate('Mean baseline', DummyRegressor(strategy='mean'))


## 6. Regression trees
A regression tree asks a sequence of questions about the predictors. Each split separates observations into two groups; the process continues until a stopping rule is reached. A new observation follows the same questions to a terminal leaf, whose training mean becomes its predicted sales.

The cell below compares maximum depths of 3, 6, and 12. `max_depth` limits the number of splits along a path from the root to a leaf. `min_samples_leaf=30` requires at least 30 training observations in each terminal leaf. Hold this second setting fixed to isolate the effect of depth.

Deeper trees can capture more detailed patterns, but may also fit random variation. Does training MAE improve more than validation MAE as depth increases? The actual depth can be below the specified maximum if other stopping rules prevent further splitting.


In [ ]:
MAX_DEPTHS = [3, 6, 12]
MIN_LEAF = 30
assert MAX_DEPTHS and all(isinstance(d, int) and d > 0 for d in MAX_DEPTHS)
assert isinstance(MIN_LEAF, int) and MIN_LEAF > 0

# Remove previous tree runs when this cell is rerun; retain the baseline.
for name in list(models):
    if name.startswith('Tree ('):
        models.pop(name)
        results.pop(name)

for depth in MAX_DEPTHS:
    name = f'Tree (max_depth={depth})'
    pipeline = fit_and_evaluate(name, DecisionTreeRegressor(
        max_depth=depth, min_samples_leaf=MIN_LEAF,
        criterion='squared_error', random_state=SEED))
    fitted_tree = pipeline.named_steps['model']
    results[name]['Actual depth'] = fitted_tree.get_depth()
    results[name]['Leaves'] = fitted_tree.get_n_leaves()


## 7. Compare validation results and select depth
Compare each tree with the baseline. A positive improvement percentage means lower MAE than the baseline; a negative value means worse performance. The code selects the tree with the lowest validation MAE. If there is a tie, the shallower maximum depth is preferred.

A wide gap between training and validation MAE can indicate overfitting or a change over time. Better training performance alone does not establish a better model. If no tree beats the baseline on validation data, retain the baseline as the preferred approach.

Before opening the test set, explain which depth performed best, how its training and validation errors differ, and whether the additional leaves improved prediction. If you change `MAX_DEPTHS` or `MIN_LEAF`, rerun sections 6–8 before the final test.


In [ ]:
assert 'Mean baseline' in results, 'Run the baseline cell first.'
tree_names = [name for name in results if name.startswith('Tree (')]
assert tree_names, 'Run the regression tree exercise first.'

comparison = pd.DataFrame(results.values()).sort_values(
    'Validation MAE', kind='stable').reset_index(drop=True)
baseline_mae = results['Mean baseline']['Validation MAE']
comparison['MAE improvement vs baseline (%)'] = (
    100 * (1 - comparison['Validation MAE'] / baseline_mae)
    if baseline_mae > 0 else np.nan)
display(comparison.round(3).fillna('—'))

selected_name = min(tree_names, key=lambda name: (
    results[name]['Validation MAE'], models[name].named_steps['model'].max_depth))
selected_depth = models[selected_name].named_steps['model'].max_depth
preferred_name = (selected_name if results[selected_name]['Validation MAE'] < baseline_mae
                  else 'Mean baseline')
print('Selected regression tree:', selected_name)
print('Preferred approach using validation MAE:', preferred_name)

comparison.set_index('Model')[['Train MAE', 'Validation MAE']].plot.barh(
    color=['#A7C4DE', '#356DA8'])
plt.xlabel('MAE: units per store/category/day')
plt.title('Training and validation errors by tree depth')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 8. Interpret the selected regression tree
The diagram shows the first three levels of the selected tree fitted on the training set. Branches below the displayed levels are omitted; a displayed node is not necessarily a terminal leaf.

Read each split from the top down. If the condition is true, follow the left branch; otherwise, follow the right branch. For a one-hot feature such as `store_id_1`, a value of 1 means the observation belongs to that store. A split at 0.5 separates membership from nonmembership. `day_of_year` is unscaled, so its threshold is a calendar-day number.

Within each node, `samples` is the number of training observations and `value` is their mean unit sales. At a terminal leaf, `value` is the prediction. Choose a displayed path and explain what it tells a store manager about how the model groups observations. These splits describe predictive patterns, not causal effects.


In [ ]:
assert 'selected_name' in globals(), 'Run section 7 first.'
selected_pipeline = models[selected_name]
selected_tree = selected_pipeline.named_steps['model']
feature_names = selected_pipeline.named_steps['prepare'].get_feature_names_out()
# Remove transformer prefixes from diagram labels for readability.
feature_names = [name.split('__', 1)[-1] for name in feature_names]

print('Training tree:', selected_name,
      '| Actual depth:', selected_tree.get_depth(),
      '| Terminal leaves:', selected_tree.get_n_leaves())
plt.figure(figsize=(18, 8))
plot_tree(selected_tree, max_depth=2, feature_names=feature_names,
          filled=True, rounded=True, impurity=False, precision=1, fontsize=10)
plt.title(f'{selected_name}: first three levels of the training tree')
plt.tight_layout()
plt.show()


## 9. Final test: open only after selection
The switch below is off by default. After selecting depth, change it to `True`. The code refits the selected tree configuration on training plus validation data and evaluates it on the untouched test period. It also refits the mean-sales baseline on the same data for comparison. The depth choice uses validation results only.

Refitting can change the tree's splits and leaf values. The diagram in section 8 shows the tree learned from the training period, before this refit.

Do not tune settings after seeing these test results. If you do, the period becomes additional validation data and you need a new test period. Predictions are left unrounded for evaluation; fractional values represent expected units.

Use the output to complete this interpretation: “With a maximum depth of ___, the tree's test MAE was ___ units per store/category/day, compared with ___ for the baseline. For replenishment planning, this means ___.” These predictions describe synthetic sales patterns; they do not establish causal effects of business decisions.


In [ ]:
RUN_FINAL_TEST = False        # Change to True only after selecting depth in section 7.

if RUN_FINAL_TEST:
    assert 'selected_name' in globals(), 'Run section 7 to select depth first.'
    assert selected_name in models, 'Rerun section 7 after changing tree settings.'
    X_development = pd.concat([X_train, X_val])
    y_development = pd.concat([y_train, y_val])

    final_tree = clone(models[selected_name]).fit(X_development, y_development)
    test_predictions = final_tree.predict(X_test)
    final_baseline = DummyRegressor(strategy='mean').fit(X_development, y_development)
    baseline_predictions = final_baseline.predict(X_test)

    print('Tree setting selected using validation MAE:', selected_name)
    print('Approach preferred before opening the test set:', preferred_name)
    test_results = pd.DataFrame([
        {'Model': selected_name, **metrics(y_test, test_predictions)},
        {'Model': 'Refitted mean baseline', **metrics(y_test, baseline_predictions)}
    ])
    display(test_results.round(3))

    prediction_examples = test[['date', 'store_id', 'category', TARGET]].copy()
    prediction_examples['predicted_units'] = test_predictions
    prediction_examples['error_actual_minus_predicted'] = y_test.to_numpy() - test_predictions
    display(prediction_examples.head(10).round(2))

    plt.scatter(y_test, test_predictions, alpha=0.15, s=12, color='#356DA8')
    low = min(y_test.min(), test_predictions.min())
    high = max(y_test.max(), test_predictions.max())
    plt.plot([low, high], [low, high], color='#B64B44', linestyle='--')
    plt.xlabel('Actual units')
    plt.ylabel('Predicted units')
    plt.title(f'Final test: {selected_name}')
    plt.tight_layout()
    plt.show()
else:
    print('Test results remain unopened. Compare validation results and select depth first.')
